# Lecture: Integrating GNNs and LLMs

## Notebook : Preparing the Cora dataset

### Xavier Bresson


In [1]:
# For Google Colaboratory
import sys, os
if "google.colab" in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/08_LLMs_GNNs'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd


## Overview

This notebook is for students who are interested in the data pipeline construction.

Its final output is the single dataset file used by the main lab:

```text
dataset/cora/cora_prepared.pt
```

Starting from the original TAPE Cora archive, we will:

1. download the archive to a temporary directory,
2. read `cora.content`, `cora.cites`, and the title/abstract extraction files,
3. keep only the 2,708 papers that belong to the Cora graph,
4. align text, node features, labels, and graph nodes by paper ID,
5. construct an undirected citation graph with self-loops,
6. row-normalize the standard bag-of-words features,
7. create one reproducible train/validation/test split,
8. export one compact dataset-ready PyTorch file,
9. reload and verify the exported file.

The large raw extraction tree is never permanently unpacked.


## Setup

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('requests') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests>=2.31'])
else:
    print('requests already installed')
    

requests already installed


### Imports and seed

We use a fixed seed so the exported train/validation/test split is reproducible.

In [3]:
import io
import random
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Publicly hosted standard Cora graph and TAPE author's text dataset
LINQS_CORA_URL = 'https://linqs-data.soe.ucsc.edu/public/lbc/cora.tgz'
GITHUB_CORA_BASE_URL = 'https://raw.githubusercontent.com/tkipf/pygcn/master/data/cora'
TAPE_CORA_BASE_URL = 'https://huggingface.co/datasets/xxhe/tape-cora/resolve/main'

DATA_DIR = Path('./dataset')
COMPACT_DIR = DATA_DIR / 'cora'
CONTENT_PATH = COMPACT_DIR / 'cora.content'
CITES_PATH = COMPACT_DIR / 'cora.cites'
TEXT_PATH = COMPACT_DIR / 'cora_text.csv'
PREPARED_DATA_PATH = COMPACT_DIR / 'cora_prepared.pt'

DATA_DIR.mkdir(parents=True, exist_ok=True)
COMPACT_DIR.mkdir(parents=True, exist_ok=True)

print('Prepared dataset will be written to:', PREPARED_DATA_PATH)


Prepared dataset will be written to: dataset/cora/cora_prepared.pt


## Read the raw archive without extracting tens of thousands of files

TAPE's archive contains the Cora graph files together with a much larger raw-text tree. We only need:

- `cora.content`,
- `cora.cites`,
- the paper-ID-to-filename mapping,
- the extraction file for each of the 2,708 Cora papers.

The helper below finds the relevant ZIP members and extracts each paper's title and abstract in memory.


In [4]:
# Download a public file and return its contents as bytes, checking for HTTP errors and empty responses
def get_public_file(url):
    # A finite timeout and HTTP status check give useful errors in Colab.
    response = requests.get(url, timeout=(20, 180))
    response.raise_for_status()
    if not response.content:
        raise RuntimeError(f'Empty download: {url}')
    return response.content

# Download the Cora graph files from LINQS, falling back to a public GitHub mirror if needed
def download_cora_graph_files():
    # LINQS is the original public distribution of the standard Cora graph.
    try:
        archive = get_public_file(LINQS_CORA_URL)
        with tarfile.open(fileobj=io.BytesIO(archive), mode='r:gz') as tf:
            for name, output_path in [('cora.content', CONTENT_PATH), ('cora.cites', CITES_PATH)]:
                matches = [member for member in tf.getmembers()
                           if member.isfile() and member.name.endswith('/' + name)]
                if len(matches) != 1:
                    raise RuntimeError(f'Expected one {name} inside the LINQS archive.')
                member_file = tf.extractfile(matches[0])
                if member_file is None:
                    raise RuntimeError(f'Cannot read {name} from the LINQS archive.')
                output_path.write_bytes(member_file.read())
        print('Cora graph source: LINQS archive')
    except (requests.RequestException, tarfile.TarError, RuntimeError, OSError) as exc:
        print(f'LINQS download unavailable ({exc}); trying the public GitHub mirror.')
        for name, output_path in [('cora.content', CONTENT_PATH), ('cora.cites', CITES_PATH)]:
            output_path.write_bytes(get_public_file(f'{GITHUB_CORA_BASE_URL}/{name}'))
        print('Cora graph source: tkipf/pygcn GitHub mirror')

# Assemble a Cora title/abstract CSV from public sources after validating graph-node and label alignment
def build_compact_files_from_public_sources():
    download_cora_graph_files()
    content = np.genfromtxt(CONTENT_PATH, dtype=str)
    if content.shape != (2708, 1435):
        raise RuntimeError(f'Unexpected cora.content shape: {content.shape}')

    # All three splits together contain one text record for each of 2708 nodes
    splits = []
    for split in ('train', 'val', 'test'):
        csv_bytes = get_public_file(f'{TAPE_CORA_BASE_URL}/{split}.csv')
        splits.append(pd.read_csv(io.BytesIO(csv_bytes), keep_default_na=False))
    text_df = pd.concat(splits, ignore_index=True)
    required_columns = {'id', 'class', 'T', 'A'}
    if not required_columns.issubset(text_df.columns):
        raise RuntimeError(f'TAPE CSV is missing columns: {required_columns - set(text_df.columns)}')
    text_df['id'] = pd.to_numeric(text_df['id'], errors='raise').astype(int)
    expected_indices = set(range(len(content)))
    if len(text_df) != len(content) or set(text_df['id']) != expected_indices or text_df['id'].duplicated().any():
        raise RuntimeError('TAPE CSV row indices do not cover the 2708 Cora graph nodes exactly once.')

    # Do not silently attach the wrong paper's text to a graph node
    ordered = text_df.set_index('id').loc[range(len(content))]
    text_classes = ordered['class'].astype(str).str.strip().str.replace(' ', '_', regex=False).to_numpy()
    if not np.array_equal(text_classes, content[:, -1]):
        bad = np.flatnonzero(text_classes != content[:, -1])
        raise RuntimeError(f'TAPE/Cora label mismatch at node indices {bad[:10].tolist()}; cannot safely align texts.')

    compact_text = pd.DataFrame({
        'paper_id': content[:, 0],
        'title': ordered['T'].astype(str).to_numpy(),
        'abstract': ordered['A'].astype(str).to_numpy(),
    })
    compact_text.to_csv(TEXT_PATH, index=False)
    print('Validated title/abstract alignment for all 2708 Cora nodes.')
    

## Download public Cora data and create the three compact files

The notebook no longer depends on the permission-restricted Google Drive archive.
The compact representation remains unchanged:

```text
dataset/cora/
├── cora.content
├── cora.cites
└── cora_text.csv
```

In [5]:
print('Downloading Cora graph and TAPE title/abstract records...')
build_compact_files_from_public_sources()

print('Compact files created:')
print(' ', CONTENT_PATH)
print(' ', CITES_PATH)
print(' ', TEXT_PATH)

Cora graph source: LINQS archive
Validated title/abstract alignment for all 2708 Cora nodes.
Compact files created:
  dataset/cora/cora.content
  dataset/cora/cora.cites
  dataset/cora/cora_text.csv


## Verify the compact dataset

Cora contains 2,708 graph nodes. Each row of `cora.content` contains

$$
1\ \text{paper ID}
+1433\ \text{bag-of-words features}
+1\ \text{class label}
=1435\ \text{columns}.
$$

We also verify that the 2,708 text records have exactly the same paper IDs as `cora.content`.

In [6]:
content = np.genfromtxt(CONTENT_PATH, dtype=str)
text_df = pd.read_csv(TEXT_PATH, dtype={"paper_id": str}, keep_default_na=False)

assert content.shape == (2708, 1435), content.shape
assert len(text_df) == 2708, len(text_df)
assert text_df["paper_id"].nunique() == 2708
assert set(text_df["paper_id"]) == set(content[:, 0])

print("Compact dataset verified.")
print("cora.content shape:", content.shape)
print("text records:", len(text_df))


Compact dataset verified.
cora.content shape: (2708, 1435)
text records: 2708


## Build graph features, labels and the reproducible split

For node $i$, the standard Cora bag-of-words feature vector is row-normalized as

$$
\tilde{\mathbf{x}}_i
=
\frac{\mathbf{x}_i}
{\max\!\left(1,\sum_j x_{ij}\right)}.
$$

Every citation is inserted in both directions, duplicate edges are removed, and every node receives a self-loop.

For $n$ nodes, the split is

$$
n_{\text{train}}=\lfloor 0.10n\rfloor,
\qquad
n_{\text{val}}=\lfloor 0.40n\rfloor,
\qquad
n_{\text{test}}=n-n_{\text{train}}-n_{\text{val}}.
$$

In [7]:
# Load the Cora paper IDs and convert the bag-of-words features into a PyTorch tensor
content = np.genfromtxt(CONTENT_PATH, dtype=str)
paper_ids = content[:, 0]
feature_values = content[:, 1:-1].astype(np.float32)
x_standard = torch.tensor(feature_values, dtype=torch.float32)

class_names = [
    "Case_Based",
    "Genetic_Algorithms",
    "Neural_Networks",
    "Probabilistic_Methods",
    "Reinforcement_Learning",
    "Rule_Learning",
    "Theory",
]

# Map the seven class names to integer labels and convert them into a target tensor
class_to_idx = {name: i for i, name in enumerate(class_names)}
label_indices = [class_to_idx[label] for label in content[:, -1]]
y = torch.tensor(label_indices, dtype=torch.long)

# Map paper IDs to node indices and convert citations into bidirectional graph edges
id_to_idx = {paper_id: i for i, paper_id in enumerate(paper_ids)}
raw_edges = np.genfromtxt(CITES_PATH, dtype=str)
src = []
dst = []
for a, b in raw_edges:
    if a in id_to_idx and b in id_to_idx:
        ia = id_to_idx[a]
        ib = id_to_idx[b]
        src.extend([ia, ib])
        dst.extend([ib, ia])

# Build the same undirected, deduplicated edges with one self-loop per node,
# without requiring DGL (whose wheels may not support the default Colab Python runtime).
num_nodes = len(paper_ids)
node_indices = torch.arange(num_nodes, dtype=torch.int64)
edge_pairs = torch.stack((torch.tensor(src, dtype=torch.int64),
                          torch.tensor(dst, dtype=torch.int64)), dim=1)
self_loops = torch.stack((node_indices, node_indices), dim=1)
edge_pairs = torch.unique(torch.cat((edge_pairs, self_loops), dim=0), dim=0)
prepared_src = edge_pairs[:, 0].contiguous()
prepared_dst = edge_pairs[:, 1].contiguous()

# Row-normalize each node's bag-of-words feature vector so its entries sum to one
row_sum = x_standard.sum(dim=1, keepdim=True).clamp(min=1.0)
x_standard = x_standard / row_sum

# Create a reproducible random 10%/40%/50% train/validation/test split using boolean masks
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(num_nodes, generator=generator)
n_train = int(0.10 * num_nodes)
n_val = int(0.40 * num_nodes)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

print(f"Graph(num_nodes={num_nodes}, num_edges={len(prepared_src)})")
print("standard features:", tuple(x_standard.shape))
print("train/val/test:", train_mask.sum().item(), val_mask.sum().item(), test_mask.sum().item())


Graph(num_nodes=2708, num_edges=13264)
standard features: (2708, 1433)
train/val/test: 270 1083 1355


## Align each graph node with its title and abstract

The alignment key is the original Cora `paper_id`. For every node $i$ we require

$$
\text{text}_i
\longleftrightarrow
\mathbf{x}^{\text{BoW}}_i
\longleftrightarrow
y_i
\longleftrightarrow
\text{graph node }i.
$$

This prevents a text embedding from being attached to the wrong paper.

In [8]:
# Load the paper text data and align each title and abstract with the graph-node order
text_df = pd.read_csv(TEXT_PATH, dtype={"paper_id": str}, keep_default_na=False)
text_by_id = text_df.set_index("paper_id")
titles = []
abstracts = []
for pid in paper_ids:
    titles.append(text_by_id.loc[pid, "title"])
    abstracts.append(text_by_id.loc[pid, "abstract"])

# Combine each paper's title and abstract into the text string used by the language model
texts = []
for title, abstract in zip(titles, abstracts):
    texts.append(f"Title: {title}\nAbstract: {abstract}")

assert len(texts) == num_nodes == 2708
assert x_standard.shape == (2708, 1433)

print("Number of aligned texts:", len(texts))
print("\nExample node 0:\n")
print(texts[0][:800])


Number of aligned texts: 2708

Example node 0:

Title: The megaprior heuristic for discovering protein sequence patterns
Abstract: Several computer algorithms for discovering patterns in groups of protein sequences are in use that are based on fitting the parameters of a statistical model to a group of related sequences. These include hidden Markov model (HMM) algorithms for multiple sequence alignment, and the MEME and Gibbs sampler algorithms for discovering motifs. These algorithms are sometimes prone to producing models that are incorrect because two or more patterns have been combined. The statistical model produced in this situation is a convex combination (weighted average) of two or more different models. This paper presents a solution to the problem of convex combinations in the form of a heuristic based on using extremely low 


## Export one dataset-ready file for the main lab

We prepare the final graph edges, features, labels, masks, paper IDs, class names, and aligned text into one PyTorch file.


In [9]:
prepared = {}
prepared["version"] = 1
prepared["num_nodes"] = num_nodes
prepared["src"] = prepared_src.cpu()
prepared["dst"] = prepared_dst.cpu()
prepared["x_standard"] = x_standard.cpu()
prepared["y"] = y.cpu()
prepared["train_mask"] = train_mask.cpu()
prepared["val_mask"] = val_mask.cpu()
prepared["test_mask"] = test_mask.cpu()
prepared["texts"] = texts
prepared["paper_ids"] = paper_ids.tolist()
prepared["class_names"] = class_names

torch.save(prepared, PREPARED_DATA_PATH)

size_mb = PREPARED_DATA_PATH.stat().st_size / 1024**2
print("Saved:", PREPARED_DATA_PATH)
print(f"Size: {size_mb:.2f} MB")

Saved: dataset/cora/cora_prepared.pt
Size: 17.42 MB
